# KLA Image Restoration — Training NotebookThis notebook pulls the code straight from GitHub, so **you never upload azip again**. When the code changes, re-run cell 1 and it fetches the latest.**Setup, once:**1. Right panel -> *Add Data* -> attach `soumikrakshit/div2k-high-resolution-images`   and your own `kla-train` dataset (the KLA pairs).2. Right panel -> *Settings* -> turn **Internet ON** (required for `git clone`).3. Accelerator -> **GPU T4 x2**. Do **not** use P100: current PyTorch has no   kernels for its sm_60 architecture and every CUDA call fails.Then: run cell 1, cell 2 (smoke test), cell 3 (data check), cell 4 (train).

## Cell 1 — copy our Phase 0/2 code into the notebookUpload the `phase0/` and `phase2/` folders as a small Kaggle Dataset called`kla-code` (a few KB), or paste the files in directly. This cell just putsthem on the import path.

In [ ]:
# ---------------------------------------------------------------------------# Pull the latest code from GitHub. No dataset upload, no waiting for Kaggle# to process a zip -- a push on the laptop is live here the moment you re-run# this cell.# ---------------------------------------------------------------------------import os, sys, glob, json, subprocessREPO_URL = "https://github.com/Kushal-MR/kla-image-restoration.git"ROOT = "/kaggle/working/repo"if os.path.isdir(os.path.join(ROOT, ".git")):    subprocess.run(["git", "-C", ROOT, "fetch", "--quiet", "origin"], check=True)    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "--quiet", "origin/main"], check=True)    print("updated existing clone")else:    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, ROOT], check=True)    print("cloned fresh")rev = subprocess.run(["git", "-C", ROOT, "log", "-1", "--format=%h %s"],                     capture_output=True, text=True).stdout.strip()print("commit:", rev)# Force a re-import if these modules were already loaded in this session,# otherwise Python keeps the stale copy and the pull appears to do nothing.for m in [k for k in list(sys.modules) if k.split(".")[0] in          ("degrade", "dataset", "model", "nafnet_sr")]:    del sys.modules[m]SRC = os.path.join(ROOT, "src")if SRC not in sys.path:    sys.path.insert(0, SRC)CFG   = os.path.join(ROOT, "configs", "degradation_config.json")SPLIT = os.path.join(ROOT, "configs", "split.json")d = sorted(glob.glob("/kaggle/input/**/DIV2K_train_HR", recursive=True))DIV2K = d[0] if d else Noneprint("DIV2K :", DIV2K, "->", len(glob.glob(str(DIV2K) + "/**/*.png", recursive=True)), "photos")KLA = next((os.path.dirname(c) for c in glob.glob("/kaggle/input/**/GT", recursive=True)), None)print("KLA   :", KLA, "->", len(glob.glob(str(KLA) + "/GT/*.npy")), "GT /",      len(glob.glob(str(KLA) + "/NoisyLR/*.npy")), "NoisyLR")assert DIV2K and KLA, "a dataset is missing -- check the Input panel"print("\nall inputs found.")

## Cell 2 — SMOKE TEST (run this first, on Accelerator = None)Twenty seconds, no GPU quota. It checks the model builds, produces exactlydouble-size output, survives odd input sizes and negative pixels, and that atraining step actually produces gradients.**If anything here fails, fix it before touching the GPU.** Burning quota on acrash from a typo is the most avoidable way to lose hours.

In [ ]:
from model.nafnet_sr import smoke_testsmoke_test("cpu")

## Cell 3 — check the synthetic data actually looks like KLA'sThe whole Phase 2 strategy rests on our fake damage matching their realdamage. Look at the pictures and the numbers before spending GPU hours on it.

In [ ]:
import numpy as np, matplotlib.pyplot as pltfrom degrade import Degrader, minmax, to_greyfrom PIL import Imagecfg = json.load(open(CFG)); print(json.dumps(cfg, indent=2))D = Degrader(CFG); rng = np.random.default_rng(0)# one synthetic pair: a DIV2K photo put through our fitted recipephoto = to_grey(np.asarray(Image.open(sorted(glob.glob(DIV2K + "/**/*.png", recursive=True))[0]),                           dtype=np.float32) / 255.0)gt_s = minmax(photo[:256, :256])lr_s = D(gt_s, rng)# one real pair from KLA, for comparisongt_r = np.load(sorted(glob.glob(KLA + "/GT/*.npy"))[0])lr_r = np.load(sorted(glob.glob(KLA + "/NoisyLR/*.npy"))[0])fig, ax = plt.subplots(2, 2, figsize=(9, 9))for a, im, t in zip(ax.ravel(), [gt_s, lr_s, gt_r, lr_r],                    ["SYNTHETIC clean", "SYNTHETIC degraded", "KLA clean", "KLA degraded"]):    a.imshow(im, cmap="gray", vmin=0, vmax=1); a.set_title(t); a.axis("off")plt.tight_layout(); plt.show()# The numbers matter more than the pictures: if these two rows disagree,# the synthetic training data is not the same problem as the real one.for name, a in [("synthetic", lr_s), ("KLA real", lr_r)]:    print(f"{name:10s} std {a.std():.3f}  max {a.max():.2f}  "          f"min {a.min():+.3f}  %neg {(a < 0).mean() * 100:.2f}")

## Cell 4 — trainSwitch **Accelerator -> GPU T4 x2**, then use *Save Version -> Save & Run All*so the run survives closing the browser.The first thing it does is build a photo cache from DIV2K (about threeminutes, once per session). Then, every 100 iterations:```ep0 it100/800 loss 0.0421  0.28 s/it  (12% waiting on data)```**`% waiting on data`** is the number to watch first. If it is high, the GPUis idle and a bigger model or more epochs will not help — the data pipelineis the bottleneck. It should be well under 30%.Then, each epoch, watch the **HARD** column and `vs bicubic`. `easy` isheld-out crops of photos the model trained on and always flatters it.

In [ ]:
cmd = [    sys.executable, os.path.join(ROOT, "train.py"),    "--photos", DIV2K,    "--real", KLA,    "--cfg", CFG,    "--split", SPLIT,    "--out", "/kaggle/working",    # Convert the 800 DIV2K PNGs to a greyscale cache once (~3 min), then    # crops are read from memory-mapped files. Without this, ~80% of every    # iteration is spent decoding PNGs and the GPU sits idle.    # /kaggle/temp is scratch space -- it does not bloat the saved output.    "--photo-cache", "/kaggle/temp/photo_cache",    "--width", "32",    "--gt-size", "256",    "--batch", "16",    "--epochs", "30",    "--iters-per-epoch", "800",    "--workers", "4",    "--real-frac", "0.25",]print(" ".join(cmd))subprocess.run(cmd, check=True)

## Cell 5 — plot the training curvesThe gap between easy and hard is the thing to watch. A widening gap meansoverfitting; you would never see it from a random split.

In [ ]:
log = json.load(open("/kaggle/working/training_log.json"))ep = [r["epoch"] for r in log]fig, ax = plt.subplots(1, 2, figsize=(12, 4))ax[0].plot(ep, [r["easy_psnr"] for r in log], label="val_easy")ax[0].plot(ep, [r["hard_psnr"] for r in log], label="val_HARD", lw=2)ax[0].set_title("PSNR (dB)"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)ax[1].plot(ep, [r["easy_ssim"] for r in log], label="val_easy")ax[1].plot(ep, [r["hard_ssim"] for r in log], label="val_HARD", lw=2)ax[1].set_title("SSIM"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()best = max(log, key=lambda r: r["hard_ssim"] * 100 + r["hard_psnr"])print("best epoch on val_hard:", best)